# Extending VAE-SRO to a new element

This notebook is the worked, runnable companion to [`docs/extending.md`](extending.md): it
walks through the three steps described there —

1. **grow the arrays** `14×14×9 → N×N×9`,
2. **augment the dataset** with alloys containing the new element, and
3. **retrain** — either full retraining from scratch, or warm-starting from the existing
   125-model ensemble —

and then shows the *actual* commands used to run each retraining mode at production scale via
`scripts/train_ensemble.py`.

Everything below was validated end-to-end against the shipped `data/database` and
`data/models` (all 9-channel feature-normalisation and `cAcB` formulas are checked against the
real shipped arrays byte-for-byte in Part 1, and the shape/weight-transfer claims in Parts 2–4
were verified against a real TensorFlow 2.15.1 / Keras 2 build of this exact architecture).

**Run this notebook with the `vae-sro` conda environment as the kernel** (`conda activate
vae-sro`, see the repository root `README.md` / `environment.yml`).

> **Scope note.** Adding a *real* new element requires two external inputs this notebook cannot
> supply: (a) the new element's 9 physical binary-pair descriptors for every existing element,
> normally computed with the optional `mapal` library (see `docs/featurisation.md`), and (b) DFT
> or experimental CSRO labels for alloys containing it. Wherever those would be needed, this
> notebook uses clearly-labelled **synthetic placeholder values** so the *mechanics* — array
> surgery, shape handling, retraining — are fully demonstrated and testable without either
> external input. Swap the placeholders for real `mapal` output / real CSRO data before using the
> result for actual science.

## 0. Setup

In [ ]:
import os
import sys
import glob
import json
import tempfile

import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt

REPO_ROOT = os.path.dirname(os.getcwd())  # docs/ -> VAE-SRO/
sys.path.insert(0, REPO_ROOT)
sys.path.insert(0, os.path.join(REPO_ROOT, "scripts"))

from vae_sro.model import ELEMENTS, DEFAULT_LATENT_DIM, compile_vae, compile_trainable_vae
from vae_sro.data import (load_feature_stack, load_alloy_arrays, load_training_data,
                           load_data_info, list_model_weights, DATABASE_DIR, MODELS_DIR)
from vae_sro.features import CACB_MIN, CACB_MAX
from vae_sro.metrics import iter_present_pairs, summarise

# scripts/train_ensemble.py is not a package -- import it directly so the demo below
# reuses the *exact* helper functions the production CLI script uses, rather than a copy.
import train_ensemble as te

print("repo root:", REPO_ROOT)
print("shipped elements:", ELEMENTS)

## 1. Baseline: confirm the current 14-element pipeline

Before changing anything, load the shipped database through the real `vae_sro` package and
re-derive every formula from [`docs/featurisation.md`](featurisation.md) by hand, checking it
reproduces the shipped arrays exactly. This is the ground truth the "grow" step below has to
preserve for the 14 original elements.

In [ ]:
stack, stack_norm, info = load_feature_stack()
arr = load_alloy_arrays()
x0, Y0 = load_training_data()
elements0, alloy_names0 = load_data_info()

print("binary-feature stack:", stack.shape)
print("alloy arrays: cAcB", arr["cAcB"].shape, "SRO", arr["SRO"].shape,
      "masks", arr["binary_masks"].shape, "n_alloys", len(alloy_names0))
print("training tensors: x", x0.shape, "Y", Y0.shape)
assert elements0 == ELEMENTS

# --- re-derive the normalisation + tensor-construction formulas from docs/featurisation.md ---
xmin, xmax = np.array(info["feature_xmin"]), np.array(info["feature_xmax"])
stack_norm_check = np.zeros_like(stack)
for k in range(9):
    span = xmax[k] - xmin[k]
    stack_norm_check[k] = (stack[k] - xmin[k]) / span if span else 0.0
assert np.allclose(stack_norm_check, stack_norm), "feature normalisation formula mismatch"

cacb_norm_check = (arr["cAcB"] - CACB_MIN) / (CACB_MAX - CACB_MIN)
assert np.allclose(cacb_norm_check, arr["cAcB_norm"]), "cAcB normalisation formula mismatch"

ft = stack_norm.transpose(1, 2, 0)  # (14, 14, 9)
x_check = cacb_norm_check[:, :, :, None] * ft[None, :, :, :]
assert np.allclose(x_check, x0, atol=1e-6), "input-tensor construction formula mismatch"
assert np.allclose(Y0[:, :, :, 0], arr["SRO"] * arr["binary_masks"]), "Y construction mismatch"

print("\nAll formulas reproduce the shipped arrays exactly -- safe to build on them below.")

## 2. Growing the tensors (`docs/extending.md`, step 1)

### The even-grid gotcha

`docs/extending.md` says enlarging the grid "changes only the input/output tensor shape... hence
tweaking, not redesigning, the architecture." That's true for the model's *parameters* (Conv2D /
Conv2DTranspose kernels don't depend on the spatial size at all — see the weight-transfer check in
Part 4), but it is **not** true for the *shape round-trip* when the new grid size is odd.

The encoder's third conv layer uses `strides=2, padding="same"`, which floors/ceils the spatial
size (`14 → 7`), and the decoder's matching transpose layer uses `strides=2, padding="same"`,
which exactly *doubles* it back (`7 → 14`). That only lands back on the original size when the
grid size is **even**: for odd `N`, `ceil(N/2) * 2 != N`, so the decoder's output silently comes
out one pixel larger than the input, and training crashes on a shape mismatch between the
`(N, N, 1)` label and the `(N+1, N+1, 1)` prediction.

The cell below reproduces the arithmetic (`ceil` for the strided conv, `×2` for the strided
transpose conv) for a range of grid sizes -- verified against a real `tf.keras` build of this
exact encoder/decoder for `N` in this range.

In [ ]:
def bottleneck_and_output_size(n):
    """Replicate encoder (conv strides 1,1,2) -> decoder (transpose strides 1,2,1) arithmetic."""
    s = n
    for stride in (1, 1, 2):        # encoder_conv_{1,2,3}
        s = -(-s // stride)         # ceil division: Conv2D 'same' + strides behaviour
    out = s
    for stride in (1, 2, 1):        # decoder_conv_tran_{1,2,4}
        out = out * stride          # Conv2DTranspose 'same' behaviour: output = input * stride
    return s, out

print(f"{'N':>3}  {'bottleneck':>10}  {'decoder out':>11}   ")
for n in range(13, 19):
    bott, out = bottleneck_and_output_size(n)
    flag = "OK" if out == n else f"MISMATCH -> decodes to {out}x{out}, not {n}x{n}"
    print(f"{n:>3}  {bott:>8}x{bott:<2} {out:>9}x{out:<2}   {flag}")

def next_even(n):
    return n if n % 2 == 0 else n + 1

**Consequence:** when the new element count is odd (14 existing + 1 new = 15), round up the
grid to the next even size and leave the extra slot(s) reserved/unused (zero row+column, zero
mask everywhere). This conveniently *also* implements the doc's own tip: "pre-allocating a larger
fixed grid... means adding an element later requires no shape change at all."

### 2a. Choose the new element vocabulary + grid size

In [ ]:
NEW_ELEMENT = "El15"   # placeholder name -- stands in for a real 15th element symbol

n_old = len(ELEMENTS)                 # 14
n_grid = next_even(n_old + 1)         # 16: rounds 15 up to the next even size
n_reserved = n_grid - (n_old + 1)     # extra slots reserved for a future element

ELEMENTS_NEW = ELEMENTS + [NEW_ELEMENT] + [f"_reserved{i}" for i in range(n_reserved)]
assert len(ELEMENTS_NEW) == n_grid

print(f"{n_old} elements -> {n_old + 1} elements -> grid rounded up to {n_grid}x{n_grid} "
      f"({n_reserved} reserved slot(s))")
print(ELEMENTS_NEW)

### 2b. Grow the 9-channel feature stack

In production this is the `mapal` step from `docs/extending.md`:

```python
from mapal import alloy_features as af, miedema_calc as miedema
# compute H_chem, H_el, comp_avg VEC / spin moment, and local mismatches
# of Allen EN, n_WS, r_met, r_cov, E for every (new_element, existing_element) pair.
```

`mapal` is an optional dependency (see `requirements.txt`) not installed here, so the cell below
draws **synthetic placeholder values** (uniformly sampled within each feature's existing observed
range) to stand in for the real descriptors -- replace `new_pair_features` with real `mapal`
output for actual use.

One correctness detail that's easy to miss: `binary-feature-info.json` stores the min/max used for
normalisation as **fixed numbers observed at dataset-construction time**, not something
recomputed on the fly. Adding pairs whose raw values fall outside the old range means the
min/max — and therefore the *entire* normalised stack, old pairs included — must be recomputed.

In [ ]:
def grow_feature_stack(stack, info, new_element_pairs, n_grid):
    """Append one row+column per new/reserved slot and renormalise.

    Parameters
    ----------
    stack : (9, n_old, n_old) raw feature stack
    info : dict with 'feature_names', 'feature_xmin', 'feature_xmax' (see load_feature_stack)
    new_element_pairs : {existing_element_index: [9 raw feature values]} for the new element
    n_grid : final (possibly padded-for-parity) grid size
    """
    n_old = stack.shape[1]
    grown = np.zeros((stack.shape[0], n_grid, n_grid))
    grown[:, :n_old, :n_old] = stack
    new_idx = n_old  # the new element occupies the first newly-appended slot
    for other_idx, vals in new_element_pairs.items():
        vals = np.asarray(vals, dtype=float)
        grown[:, new_idx, other_idx] = vals
        grown[:, other_idx, new_idx] = vals   # descriptors are pair-symmetric

    xmin = np.minimum(info["feature_xmin"], grown.reshape(grown.shape[0], -1).min(axis=1))
    xmax = np.maximum(info["feature_xmax"], grown.reshape(grown.shape[0], -1).max(axis=1))
    grown_norm = np.zeros_like(grown)
    for k in range(grown.shape[0]):
        span = xmax[k] - xmin[k]
        grown_norm[k] = (grown[k] - xmin[k]) / span if span else 0.0

    new_info = dict(info)
    new_info["feature_xmin"] = xmin.tolist()
    new_info["feature_xmax"] = xmax.tolist()
    return grown, grown_norm, new_info


rng = np.random.default_rng(0)
# SYNTHETIC placeholder: one 9-vector per (NEW_ELEMENT, existing_element) pair.
new_pair_features = {i: rng.uniform(low=info["feature_xmin"], high=info["feature_xmax"])
                      for i in range(n_old)}

new_stack, new_stack_norm, new_info = grow_feature_stack(stack, info, new_pair_features, n_grid)
assert new_stack.shape == (9, n_grid, n_grid)
assert np.allclose(new_stack[:, :n_old, :n_old], stack), "existing 14x14 block must be untouched"
print("grown feature stack:", new_stack.shape, "-- original 14x14 block preserved exactly")

### 2c. Grow the existing alloys' arrays + augment the dataset (`docs/extending.md`, step 2)

Existing alloys get zero-padded rows/columns (they don't contain the new element, so their mask
stays 0 there). Then we add alloy(s) that *do* contain the new element -- in production these
come with real DFT/experimental CSRO; here the label is a placeholder zero array, clearly not a
real measurement.

In [ ]:
def pad_grid(a, n_grid):
    out = np.zeros((a.shape[0], n_grid, n_grid), dtype=a.dtype)
    out[:, :a.shape[1], :a.shape[2]] = a
    return out

cacb_grown = pad_grid(arr["cAcB"], n_grid)
sro_grown = pad_grid(arr["SRO"], n_grid)
masks_grown = pad_grid(arr["binary_masks"], n_grid)

# --- one new alloy containing NEW_ELEMENT (composition is real, SRO label is a placeholder) ---
new_comp = {"Ti": 0.4, NEW_ELEMENT: 0.2, "Cr": 0.2, "Fe": 0.1, "Ni": 0.1}
c = np.zeros(n_grid)
for el, frac in new_comp.items():
    c[ELEMENTS_NEW.index(el)] = frac
present = [ELEMENTS_NEW.index(el) for el in new_comp]
new_cacb, new_mask = np.zeros((n_grid, n_grid)), np.zeros((n_grid, n_grid))
for i in present:
    for j in present:
        if i != j:
            new_cacb[i, j] = c[i] * c[j]
            new_mask[i, j] = 1.0
new_sro = np.zeros((n_grid, n_grid))  # PLACEHOLDER -- replace with real DFT/experimental CSRO

cacb_grown = np.concatenate([cacb_grown, new_cacb[None]], axis=0)
sro_grown = np.concatenate([sro_grown, new_sro[None]], axis=0)
masks_grown = np.concatenate([masks_grown, new_mask[None]], axis=0)
alloy_names_grown = np.concatenate([arr["alloy_names"], np.array(["Ti0.4El15-0.2Cr0.2Fe0.1Ni0.1"])])

print(f"alloys: {arr['cAcB'].shape[0]} -> {cacb_grown.shape[0]} "
      f"(add every real new-element alloy here, not just one)")

### 2d. Build the final training tensors and round-trip through the real `vae_sro.data` loaders

In [ ]:
cacb_norm_grown = (cacb_grown - CACB_MIN) / (CACB_MAX - CACB_MIN)
ft_new = new_stack_norm.transpose(1, 2, 0)
x_grown = (cacb_norm_grown[:, :, :, None] * ft_new[None, :, :, :]).astype("float32")
Y_grown = (sro_grown * masks_grown)[:, :, :, None].astype("float32")
print("final training tensors:", x_grown.shape, Y_grown.shape)

# Write the extended database in the exact shipped-file layout so the *unmodified* vae_sro.data
# loaders can read it back via `database_dir=...`. Using a temp dir here so re-running this
# notebook never leaves generated binary artifacts in the repo; for a real extension, write to a
# permanent location (e.g. a new `data/database_v15/`) instead.
demo_db_dir = tempfile.mkdtemp(prefix="vae_sro_extend_demo_")
np.save(os.path.join(demo_db_dir, "binary-feature-stack.npy"), new_stack)
np.save(os.path.join(demo_db_dir, "binary-feature-stack-norm.npy"), new_stack_norm)
with open(os.path.join(demo_db_dir, "binary-feature-info.json"), "w") as f:
    json.dump(new_info, f)
np.savez(os.path.join(demo_db_dir, "alloys-data-arrays.npz"),
          cAcB=cacb_grown, cAcB_norm=cacb_norm_grown, SRO=sro_grown,
          binary_masks=masks_grown, alloy_names=alloy_names_grown,
          binary_counts=np.zeros((n_grid, n_grid)))
with open(os.path.join(demo_db_dir, "alloys-data-info.json"), "w") as f:
    json.dump({"elements": ELEMENTS_NEW, "alloy_names": alloy_names_grown.tolist(),
               "min_components": arr.get("min_components", 3),
               "cAcB_min_for_norm": CACB_MIN, "cAcB_max_for_norm": CACB_MAX}, f, default=int)
np.savez(os.path.join(demo_db_dir, "data-x-Y-for-training.npz"), x=x_grown, Y=Y_grown)

# round-trip through the real, *unmodified* package loaders:
x_rt, Y_rt = load_training_data(database_dir=demo_db_dir)
elements_rt, names_rt = load_data_info(database_dir=demo_db_dir)
assert np.allclose(x_rt, x_grown) and elements_rt == ELEMENTS_NEW
print(f"round-trip via vae_sro.data.*(database_dir=...) OK -- wrote to {demo_db_dir}")

## 3. What would need to change in `vae_sro/` for a *permanent* extension

Everything above worked without modifying the installed package, but only because it recreated a
few functions locally. A permanent extension to the shipped code should instead parameterise the
handful of spots that currently hardcode `14`:

- `vae_sro/features.py::composition_to_cacb` -- allocates `np.zeros((14, 14))` directly; should
  derive the size from `len(ELEMENTS)`.
- `vae_sro/predict.py::EnsemblePredictor.raw_predictions` -- preallocates
  `np.zeros((n_models, n_samples, len(x), 14, 14))`; same fix.
- `vae_sro/model.py::ELEMENTS` -- the vocabulary itself; append the new element(s) here (and any
  reserved placeholders), since `vae_sro.metrics`, `vae_sro.dataset`, and `vae_sro.predict` all
  import this list to label pairs.

The model architecture itself (`build_vae`, `compile_vae`, `compile_trainable_vae`) needs **no
changes** -- `input_dim` is already a parameter, and, as shown in Part 4 below, the convolutional
weights don't even depend on the grid size.

## 4. Step 3 (`docs/extending.md`): retrain

Both retraining modes below reuse `predict_masked` / `fold_metrics` from `scripts/train_ensemble.py`
directly (imported as `te` in Part 0), so the demo here is exercising the same code the production
CLI runs -- just at a tiny scale (a few epochs, one fold) for interactive speed.

### 4a. Full retraining from scratch

This is what `docs/extending.md` recommends "when data for the new element are sufficient": build
a fresh model at the new input shape and train it the normal way.

In [ ]:
x_train, Y_train = x_rt[:-5], Y_rt[:-5]
x_val, Y_val = x_rt[-5:], Y_rt[-5:]
masks_val = masks_grown[-5:]

tf.keras.backend.clear_session()
tf.random.set_seed(0)
model_scratch = compile_trainable_vae(x_rt.shape[1:], learning_rate=5e-4)
history_scratch = model_scratch.fit(x_train, Y_train, epochs=200, verbose=0)

pred_scratch = te.predict_masked(model_scratch, x_val, masks_val)
mis, mae = te.fold_metrics(np.squeeze(Y_val), pred_scratch, masks_val)
print(f"from-scratch demo (200 epochs, 1 fold): val_misclass={mis}, val_mae={mae:.4f}")

plt.figure(figsize=(5, 3))
plt.plot(history_scratch.history["loss"])
plt.xlabel("epoch"); plt.ylabel("training loss"); plt.title("From-scratch demo fit")
plt.show()

**Actual production command** (once the extended database is written to a permanent path, not
the temp dir used above), mirroring the paper's full `--shuffles 5 --runs 5 --folds 5` protocol —
this exact flag was added to `scripts/train_ensemble.py` to support training on an extended
database:

```bash
conda activate vae-sro
python scripts/train_ensemble.py \
    --database-dir data/database_v15 \
    --outdir data/models_v15_scratch \
    --shuffles 5 --runs 5 --folds 5 --epochs 2000 --save-every 100
```

### 4b. Warm start / transfer learning from the existing 125-model ensemble

Attractive when the new element's data are sparse, per `docs/extending.md`: reuse the learned
pairwise representation and only adapt what has to change.

The key mechanical fact (verified below): `Conv2D`/`Conv2DTranspose`/`BatchNormalization` kernels
have shape `(k, k, in_channels, out_channels)` — **independent of the grid's height/width** — so
they transfer directly from a 14×14-trained model into a 16×16 model. Only the three `Dense`
layers whose shape depends on the flattened bottleneck size (`encoder_mu`, `encoder_log_variance`,
`decoder_dense_1`) cannot transfer and fall back to their fresh random initialisation.
`model.load_weights(path, by_name=True, skip_mismatch=True)` does exactly this split
automatically, and warns about every skipped weight so you can see it happening.

In [ ]:
shipped_weights = list_model_weights()
print(f"{len(shipped_weights)} shipped ensemble weight files available in {MODELS_DIR}")
src_weights = shipped_weights[0]

tf.keras.backend.clear_session()
tf.random.set_seed(0)
model_ws = compile_trainable_vae(x_rt.shape[1:], learning_rate=5e-4)

# 'encoder_with_stats' is compile_trainable_vae's name for the encoder sub-model (compile_vae,
# used for inference on the *original* 14x14 shape, calls the same sub-model 'encoder_model' --
# by_name matching works either way since it keys off each leaf layer's own name, e.g.
# 'encoder_conv_2', not the outer container name).
mu_before = model_ws.get_layer("encoder_with_stats").get_layer("encoder_mu").get_weights()[0].copy()

model_ws.load_weights(src_weights, by_name=True, skip_mismatch=True)

conv_after = model_ws.get_layer("encoder_with_stats").get_layer("encoder_conv_2").get_weights()[0]
mu_after = model_ws.get_layer("encoder_with_stats").get_layer("encoder_mu").get_weights()[0]
print("\nconv weights carried over from the 14x14-trained model (not all-zero/random):",
      not np.allclose(conv_after, 0))
print("encoder_mu Dense weights left at fresh random init (shape mismatch -> skipped):",
      np.allclose(mu_before, mu_after))

history_ws = model_ws.fit(x_train, Y_train, epochs=200, verbose=0)
pred_ws = te.predict_masked(model_ws, x_val, masks_val)
mis_ws, mae_ws = te.fold_metrics(np.squeeze(Y_val), pred_ws, masks_val)
print(f"\nwarm-start demo (200 epochs, 1 fold): val_misclass={mis_ws}, val_mae={mae_ws:.4f}")

**Actual production command** — `--warm-start-dir` (also added to
`scripts/train_ensemble.py`) loads a source weight file into every fold before fine-tuning, cycling
through the given directory's files if there are fewer of them than folds to fill:

```bash
conda activate vae-sro
python scripts/train_ensemble.py \
    --database-dir data/database_v15 \
    --warm-start-dir data/models \
    --outdir data/models_v15_warmstart \
    --shuffles 5 --runs 5 --folds 5 --epochs 500 --save-every 100
```

Fewer epochs are typically needed than from-scratch training, since the convolutional feature
extractor starts from the already-trained 14-element representation.

## 5. Summary

- **Grow**: append a row/column per new element to `cAcB` and all 9 feature layers; round the
  final grid size up to the next **even** number (Part 2) — odd grid sizes break the
  encoder/decoder shape round-trip.
- **Renormalise**: the 9-channel feature stack's min/max are fixed constants computed at
  dataset-build time (`binary-feature-info.json`), not recomputed per grid — they must be
  recomputed over the enlarged stack (Part 2b).
- **Augment**: pad existing alloys with zero rows/columns/mask; add new alloys with real DFT or
  experimental CSRO labels for the new element (this notebook uses placeholders — Part 2c).
- **Retrain from scratch** (`compile_trainable_vae` at the new shape) when new-element data are
  plentiful; **warm-start** (`load_weights(..., by_name=True, skip_mismatch=True)`) when they are
  sparse — only the three bottleneck `Dense` layers fail to transfer and re-initialise (Part 4).
- Both retraining modes are runnable today via `scripts/train_ensemble.py --database-dir ... 
  [--warm-start-dir ...]`.
- For a permanent (not one-off/notebook) extension, parameterise the hardcoded `14`s in
  `vae_sro/features.py` and `vae_sro/predict.py`, and extend `vae_sro/model.py::ELEMENTS`
  (Part 3).